# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` value, following the dataset schema. This allows precise selection of record sets, fields, and columns during processing.

In [ ]:
# List all record sets (`@id`)
print("Available record sets in the dataset (referenced by @id):\n")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set['@id']} : {record_set.get('name', '')}")

# Optionally, list available fields in each record set
for record_set in dataset.metadata.record_sets:
    print(f"\nFields for record set {record_set['@id']}:")
    for field in record_set['fields']:
        print(f"  - {field['@id']}: {field.get('name', '')}")

## 3. Data Extraction
Load data from one or more record set(s) into a DataFrame for analysis. Use the record set and field `@id`s from the overview. Here, each record set is loaded dynamically using the proper `@id`.

In [ ]:
# List all record set @id's in the dataset
record_sets_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        print(f"Loaded DataFrame for {record_set_id} (shape: {df.shape})")
        dataframes[record_set_id] = df
    else:
        print(f"No data found for {record_set_id}.")

# Select the first record set for demonstration
if dataframes:
    primary_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in {primary_rs_id}:")
    print(dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())
else:
    primary_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming or grouping numeric columns, etc. Refer to each field by its `@id`.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# For this dataset, select a likely numeric field for analysis.
numeric_field_id = None
group_field_id = None

# Automatically pick the first numeric-looking field in the primary record set
if primary_rs_id is not None:
    df = dataframes[primary_rs_id]
    for col in df.columns:
        # Try to find a numeric field by sampling values
        try:
            vals = pd.to_numeric(df[col], errors='coerce')
            if vals.notnull().sum() > 0 and vals.notnull().mean() > 0.5:
                numeric_field_id = col
                break
        except:
            continue

    # Try to find a likely group field
    for col in df.columns:
        if col != numeric_field_id:
            nunique = df[col].nunique()
            if 1 < nunique < 20:
                group_field_id = col
                break

    if numeric_field_id is None:
        print("No numeric field detected for analysis.")
    else:
        print(f"Selected numeric field for EDA: '{numeric_field_id}' (as referenced by @id)")
        # Convert to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Example threshold (median for demonstration)
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered to records with {numeric_field_id} > {threshold}")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id:
            print(f"\nGrouping by '{group_field_id}':")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
else:
    print("No record sets with records loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The following example shows basic plotting for the selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_rs_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[primary_rs_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(12, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[primary_rs_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading a Croissant-structured biomedical mass spectrometry dataset with the `mlcroissant` library and performed initial exploration, extraction, and analysis steps. Fields and record sets were referenced using their `@id` as per best practice for Croissant datasets.

**Key findings and next steps:**
- The dataset contains rich protein-level data with various numeric attributes available for further analysis.
- Initial visualizations reveal the distribution of selected measurement fields; further analysis can include cross-sample comparisons, exploring modification effects, or identifying biomarkers.
- For advanced analytics, map field `@id`s to human-readable labels as needed, and carefully preprocess for downstream machine learning tasks.